# TickNet-L v1: Polishing Phase (Two-Stage Optimization)

This notebook fine-tunes the two best-performing validation-selected models:
- **CIFAR-10 Baseline:**  (**94.97%** Test Top-1)
- **CIFAR-100 Baseline:**  (**75.28%** Test Top-1)

### Polishing Enhancements Applied:
1. **Full 50,000 Dataset:** Utilizes 100% of training data ().
2. **Label Smoothing (0.05):** Mitigates overconfident predictions on confusing classes.
3. **Stochastic Weight Averaging (SWA):** Finds flat minima basins over the final 15 epochs.
4. **Cosine Annealing (25 epochs, lr=0.01 -> 0):** Low-energy refinement without disrupting converged weights.
5. **Test-Time Augmentation (Horizontal Flip TTA):** Boosts evaluation stability and accuracy.

In [ ]:
# 1. Environment & GPU Check
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device     : {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Clone repository branch and set working directory
import os, sys
from pathlib import Path

if not Path("TickNets").exists():
    !git clone -b feature/final-exam-model-l-polish https://github.com/khuonglaptri3/TickNets.git
%cd TickNets
sys.path.insert(0, os.getcwd())
!pip install -q pytest

In [ ]:
# 3. Download CIFAR datasets if not present
!python download_cifar.py --data-root data --dataset all

## Part 1: Polish CIFAR-10 Model (Target: >= 96.0%)

In [ ]:
!python polish_cifar.py \
    --checkpoint checkpoints/cifar10_sgd_lr015/best_val.pt \
    --dataset cifar10 \
    --data-root data \
    --output-dir runs/polished_cifar10 \
    --epochs 25 \
    --batch-size 128 \
    --lr 0.01 \
    --label-smoothing 0.05 \
    --use-swa \
    --swa-start-ratio 0.4 \
    --val-fraction 0.0

## Part 2: Polish CIFAR-100 Model (Target: >= 76.5%)

In [ ]:
!python polish_cifar.py \
    --checkpoint checkpoints/cifar100_sgd_lr015/best_val.pt \
    --dataset cifar100 \
    --data-root data \
    --output-dir runs/polished_cifar100 \
    --epochs 25 \
    --batch-size 128 \
    --lr 0.01 \
    --label-smoothing 0.05 \
    --use-swa \
    --swa-start-ratio 0.4 \
    --val-fraction 0.0

## Part 3: Final Comparison Summary & Archive

In [ ]:
import json
import pandas as pd

summary_rows = []
for ds in ["cifar10", "cifar100"]:
    p = Path(f"runs/polished_{ds}/polish_summary.json")
    if p.exists():
        data = json.loads(p.read_text())
        summary_rows.append({
            "Dataset": ds.upper(),
            "Baseline Top-1 (%)": f"{data['baseline_standard_top1']:.2f}%",
            "Polished Standard Top-1 (%)": f"{data['results']['Standard_Model_standard']['top1']:.2f}%",
            "Polished SWA Top-1 (%)": f"{data['results'].get('SWA_Model_standard', {}).get('top1', 0):.2f}%",
            "Polished + Flip-TTA (%)": f"{data['best_polished_top1']:.2f}%",
            "Absolute Gain": f"{data['absolute_gain']:+.2f}%",
        })

df_summary = pd.DataFrame(summary_rows)
print(df_summary.to_markdown(index=False))